# Engagement 1: Duka Electronics Sales Exploration (H1 2026)

## Executive Summary

- Duka Electronics generated KES 110,255,000 from 1,000 orders during January to June 2026.
- Nakuru led the branches with KES 24,326,000 in revenue and 207 orders.
- Laptops led product revenue at KES 52,815,000, while Smartwatches and Headphones each led units sold with 664 units.
- January was the strongest month at KES 20,238,000, compared with June at KES 16,777,000, a 20.6% difference.
- There were 82 Premium Online orders from Nairobi and Mombasa, worth KES 14,890,000 in total.

## First-five-minutes inspection

The first inspection checks the shape, column types, missing values, date range, and key categories before any transformation. This makes unexpected schema, completeness, time-window, or category problems visible early.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('duka_sales.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run python generate_data.py from the repository root first.')

sales = pd.read_csv(DATA_PATH, parse_dates=['order_date'])
print('Shape:', sales.shape)
print('Types:')
print(sales.dtypes)
print('Missing values:')
print(sales.isna().sum())
print('Date range:', sales['order_date'].min().date(), 'to', sales['order_date'].max().date())
print('Branches:', sorted(sales['branch'].unique()))
print('Products:', sorted(sales['product'].unique()))
print('Channels:', sorted(sales['channel'].unique()))

## Derived columns

In [ ]:
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales['month'] = sales['order_date'].dt.to_period('M').astype(str)
sales['weekday'] = sales['order_date'].dt.day_name()
sales['is_weekend'] = sales['order_date'].dt.dayofweek >= 5
sales['price_band'] = sales['unit_price'].ge(28000).map({True: 'Premium', False: 'Value'})
sales.head()

## Task 3: Branch rankings

Revenue and order rankings are not necessarily identical: revenue also reflects product mix and quantity, while order count gives every order equal weight.

In [ ]:
branch_summary = sales.groupby('branch').agg(
    total_revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique')
).sort_values('total_revenue', ascending=False)
branch_summary

## Task 4: Product performance

The revenue and units lists can differ because products have different unit prices. High-priced products can lead revenue even when lower-priced products sell more units.

In [ ]:
product_summary = sales.groupby('product').agg(
    revenue=('revenue', 'sum'),
    units_sold=('quantity', 'sum')
).sort_values('revenue', ascending=False)
product_summary

## Task 5: Weekend versus weekday orders

This comparison uses order count, total revenue, and average revenue per order so volume and value are considered together.

In [ ]:
weekend_summary = sales.groupby(sales['is_weekend'].map({True: 'Weekend', False: 'Weekday'})).agg(
    orders=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum'),
    average_order_revenue=('revenue', 'mean')
)
weekend_summary

## Task 6: Best and worst month

The percentage difference is calculated as `(best month - worst month) / worst month * 100`.

In [ ]:
monthly_revenue = sales.groupby('month')['revenue'].sum().sort_index()
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
percentage_difference = (monthly_revenue[best_month] - monthly_revenue[worst_month]) / monthly_revenue[worst_month] * 100
print('Best:', best_month, monthly_revenue[best_month])
print('Worst:', worst_month, monthly_revenue[worst_month])
print('Percentage difference:', round(percentage_difference, 1), '%')
monthly_revenue

## Task 7: Premium online coastal-capital orders

The requested filter keeps Premium orders sold Online by the Nairobi or Mombasa branches and exports the complete filtered rows without a pandas index.

In [ ]:
premium_online_coast_capital = sales.loc[
    sales['price_band'].eq('Premium')
    & sales['channel'].eq('Online')
    & sales['branch'].isin(['Nairobi', 'Mombasa'])
].copy()
premium_online_coast_capital.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print('Orders:', len(premium_online_coast_capital))
print('Total revenue:', premium_online_coast_capital['revenue'].sum())
premium_online_coast_capital.head()

## Task 8: Spreadsheet reproduction

The workbook contains a SUMIFS branch revenue table, an XLOOKUP region column, and a PivotTable-style revenue summary by branch and channel. The numeric totals are checked against the pandas result before saving.

In [ ]:
from pathlib import Path

region_map = pd.DataFrame({
    'branch': ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret'],
    'region': ['Capital', 'Coast', 'Rift Valley', 'Nyanza', 'Rift Valley']
})
branch_order = sales.groupby('branch')['revenue'].sum().sort_values(ascending=False).index.tolist()
branch_table = pd.DataFrame({'branch': branch_order})
branch_table['pandas_revenue'] = branch_table['branch'].map(sales.groupby('branch')['revenue'].sum())
branch_table['orders'] = branch_table['branch'].map(sales.groupby('branch')['order_id'].nunique())
branch_table['SUMIFS_revenue'] = [f'=SUMIFS(SalesData!$G:$G,SalesData!$C:$C,A{row})' for row in range(3, 3 + len(branch_table))]
branch_table['region_XLOOKUP'] = [f'=XLOOKUP(A{row},Regions!$A$2:$A$6,Regions!$B$2:$B$6,"Unknown")' for row in range(3, 3 + len(branch_table))]
pivot_table = sales.pivot_table(index='branch', columns='channel', values='revenue', aggfunc='sum', fill_value=0, margins=True, margins_name='Grand Total')
Path('excel').mkdir(exist_ok=True)
with pd.ExcelWriter('excel/duka_branch_summary.xlsx') as writer:
    sales.to_excel(writer, sheet_name='SalesData', index=False)
    region_map.to_excel(writer, sheet_name='Regions', index=False)
    branch_table.to_excel(writer, sheet_name='BranchSummary', index=False, startrow=1)
    pivot_table.to_excel(writer, sheet_name='PivotTable')

print('Pandas total:', sales['revenue'].sum())
print('Pivot total:', pivot_table.loc['Grand Total', 'Grand Total'])
print('Totals match:', sales['revenue'].sum() == pivot_table.loc['Grand Total', 'Grand Total'])

## Task 9: Executive Summary

The five numbered findings at the top are the executive summary: they give leadership the overall revenue, leading branch, product results, monthly comparison, and requested filtered-order result with specific numbers.